In [17]:
import pandas as pd
import numpy as np

#LOAD DATA

df = pd.read_excel("Book2.xlsx")

# Fix header
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

df.columns = [
    "Name","Age","Gravida","TT","Gestation","Weight",
    "Height","BloodPressure","Anemia","Jaundice",
    "FetalPosition","FetalMovement","FetalHeartRate",
    "Albumin","Sugar","VDRL","HBsAg","Risk"
]

# CLEAN NUMERIC DATA

# Age
df["Age"] = pd.to_numeric(df["Age"], errors='coerce')

# Gravida (1st → 1)
df["Gravida"] = df["Gravida"].astype(str).str.replace(r'\D', '', regex=True)
df["Gravida"] = pd.to_numeric(df["Gravida"], errors='coerce')

# TT (fix the error here)
df["TT"] = df["TT"].astype(str).str.replace(r'\D', '', regex=True)
df["TT"] = pd.to_numeric(df["TT"], errors='coerce')

# Gestation (38 week → 38)
df["Gestation"] = df["Gestation"].astype(str).str.replace(" week","")
df["Gestation"] = pd.to_numeric(df["Gestation"], errors='coerce')

# Weight (50 kg → 50)
df["Weight"] = df["Weight"].astype(str).str.replace(" kg","")
df["Weight"] = pd.to_numeric(df["Weight"], errors='coerce')

# Height (5.3'' → 5.3)
df["Height"] = df["Height"].astype(str).str.replace("''","")
df["Height"] = pd.to_numeric(df["Height"], errors='coerce')

# Blood Pressure → split
bp = df["BloodPressure"].astype(str).str.split("/", expand=True)
df["SystolicBP"] = pd.to_numeric(bp[0], errors='coerce')
df["DiastolicBP"] = pd.to_numeric(bp[1], errors='coerce')

# Fetal Heart Rate (140m → 140)
df["FetalHeartRate"] = df["FetalHeartRate"].astype(str).str.replace("m","")
df["FetalHeartRate"] = pd.to_numeric(df["FetalHeartRate"], errors='coerce')

# HANDLE CATEGORICALS

binary_map = {
    "Yes": 1, "No": 0,
    "Positive": 1, "Negative": 0,
    "Normal": 0
}

cat_cols = ["Anemia","Jaundice","FetalPosition","FetalMovement",
            "Albumin","Sugar","VDRL","HBsAg"]

for col in cat_cols:
    df[col] = df[col].map(binary_map)

# Encode target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["Risk"] = le.fit_transform(df["Risk"])

# Drop unused columns
df = df.drop(columns=["Name","BloodPressure"])

# FORCE EVERYTHING NUMERIC (prevents ALL errors)
df = df.apply(pd.to_numeric, errors='coerce')

# Fill missing
df = df.fillna(0)

# Debug check
print("\nData Types:\n", df.dtypes)

# TRAIN MODEL
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

X = df.drop("Risk", axis=1)
y = df["Risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# EVALUATE
preds = model.predict(X_test)
accuracy = accuracy_score(y_test, preds)

print("\nAccuracy:", accuracy)

# SAVE MODEL

import joblib

joblib.dump(model, "xgboost_model.pkl")
joblib.dump(le, "label_encoder.pkl")

print("\nModel saved successfully!")


Data Types:
 Age                 int64
Gravida             int64
TT                  int64
Gestation           int64
Weight              int64
Height            float64
Anemia            float64
Jaundice          float64
FetalPosition     float64
FetalMovement       int64
FetalHeartRate      int64
Albumin           float64
Sugar               int64
VDRL                int64
HBsAg               int64
Risk                int64
SystolicBP          int64
DiastolicBP         int64
dtype: object

Accuracy: 0.97

Model saved successfully!
